# Continuity (Concept 11)

**Prerequisite:** sequences and limits (concept 10).

This notebook walks through the $\epsilon$-$\delta$ definition of continuity, the sequential characterisation, and Lipschitz continuity. We verify everything numerically with the Python standard library only.


## 1. The $\epsilon$-$\delta$ definition

$f : D \to \mathbb{R}$ is **continuous at** $c \in D$ iff
$$\forall \epsilon > 0,\ \exists \delta > 0,\ \forall x \in D,\quad |x - c| < \delta \implies |f(x) - f(c)| < \epsilon.$$

We will verify this for $f(x) = x^2$ at $c = 2$. The key inequality is
$$|x^2 - 4| = |x - 2|\,|x + 2|.$$
For $|x - 2| < 1$ we have $|x + 2| < 5$, so choosing $\delta = \min(1, \epsilon/5)$ certifies the implication.


In [ ]:
def delta_for_x2_at_2(eps):
    return min(1.0, eps / 5.0)

def verify(eps, n=400):
    d = delta_for_x2_at_2(eps)
    worst = 0.0
    for k in range(1, n):
        x = 2.0 - d + 2.0 * d * k / n
        worst = max(worst, abs(x*x - 4.0))
    return d, worst, worst < eps

for eps in [1.0, 1e-1, 1e-2, 1e-4, 1e-6]:
    d, w, ok = verify(eps)
    print(f"eps={eps:<8g} -> delta={d:<10.6f}  max|f(x)-f(2)|={w:<12.6g} pass={ok}")


## 2. Sequential characterisation and failure

**Theorem.** $f$ is continuous at $c$ iff $x_n \to c \implies f(x_n) \to f(c)$.

We exhibit failure for the Dirichlet indicator $\mathbf{1}_\mathbb{Q}$ at $c = \sqrt{2}$:
rational truncations $x_n = \lfloor 10^n \sqrt{2}\rfloor / 10^n$ satisfy $x_n \to \sqrt{2}$ and $\mathbf{1}_\mathbb{Q}(x_n) = 1$,
but $\mathbf{1}_\mathbb{Q}(\sqrt{2}) = 0$. So $\mathbf{1}_\mathbb{Q}$ cannot be continuous at $\sqrt{2}$.


In [ ]:
from fractions import Fraction
from math import sqrt

s = sqrt(2.0)
for n in range(1, 9):
    x_n = Fraction(int(s * 10**n), 10**n)
    print(f"n={n}  x_n={float(x_n):.8f}  1_Q(x_n)=1   (rational)")
print(f"limit = sqrt(2) ~ {s:.8f}  1_Q(sqrt(2)) = 0   (irrational)")
print("sequential characterisation fails -> discontinuous at sqrt(2).")


## 3. Lipschitz continuity

$f$ is **Lipschitz with constant $L$** on $D$ if
$$|f(x) - f(y)| \leq L\,|x - y| \quad \text{for all } x, y \in D.$$

Lipschitz $\Rightarrow$ uniformly continuous (take $\delta = \epsilon / L$). The best (smallest) Lipschitz constant is
$$L^* = \sup_{x \ne y} \frac{|f(x) - f(y)|}{|x - y|}.$$

For differentiable $f$ on a convex domain, $L^* = \sup |f'|$. Examples:
- $f(x) = \sin x$ on $\mathbb{R}$: $L^* = 1$.
- $f(x) = x^2$ on $[-M, M]$: $L^* = 2M$ (so $x^2$ is *not* globally Lipschitz on $\mathbb{R}$).


In [ ]:
import math, random
random.seed(0)

def estimate_L(f, lo, hi, n=20000):
    L = 0.0
    for _ in range(n):
        x = random.uniform(lo, hi)
        y = random.uniform(lo, hi)
        if x != y:
            L = max(L, abs(f(x) - f(y)) / abs(x - y))
    return L

print(f"sin on [-10, 10]:   estimated L = {estimate_L(math.sin, -10, 10):.4f}  (theory: 1)")
print(f"x^2 on [-3, 3]:     estimated L = {estimate_L(lambda x: x*x, -3, 3):.4f}  (theory: 6)")
print(f"x^2 on [-100,100]:  estimated L = {estimate_L(lambda x: x*x, -100, 100):.4f}  (theory: 200)")


## 4. Exercises

1. **$1/x$ is not uniformly continuous on $(0, 1]$.** Adapt the rational-approximation construction above: take $x_n = 1/n$, $y_n = 1/(n+1)$. Show $|x_n - y_n| \to 0$ while $|f(x_n) - f(y_n)| = 1$, so uniformity fails.
2. **Heine--Cantor in practice.** Verify numerically that $f(x) = \sqrt{x}$ on $[0, 1]$ admits a single $\delta(\epsilon)$ working for the entire interval (find such a $\delta$ for $\epsilon = 0.01$). Hint: use $|\sqrt{x} - \sqrt{y}| \leq \sqrt{|x - y|}$.
3. **Lipschitz constants and gradient descent.** For $f(x) = (x - 3)^2$ with $L = 2$ (Lipschitz constant of $f'$), run gradient descent with step sizes $\eta = 1/L$, $\eta = 1.5/L$, $\eta = 2.5/L$ from $x_0 = 0$. Report which step sizes converge and tie this to the descent lemma.
